In [1]:
pip install opencv-python numpy matplotlib

Note: you may need to restart the kernel to use updated packages.


You should consider upgrading via the 'C:\Users\abrah\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.9_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip' command.


In [25]:
import cv2
import numpy as np
import math

# Variables globales
points = []          # Puntos seleccionados por el usuario
mesh_triangles = []  # Lista de triángulos generados
obj_center = None    # Centro del objeto detectado

# Función para generar triángulos equiláteros adyacentes
def generate_equilateral_mesh(p1, p2, p3, rows=5, cols=5):
    triangles = []
    base_triangle = np.array([p1, p2, p3], dtype=np.float32)
    triangles.append(base_triangle)

    v1 = p2 - p1  # Vector base
    v2 = p3 - p1  # Vector lateral

    for i in range(-rows, rows + 1):
        for j in range(-cols, cols + 1):
            dx = i * v1 + j * v2
            new_triangle = base_triangle + dx
            triangles.append(new_triangle)

    return triangles

# Función para saber en qué triángulo está un punto
def point_in_triangle(pt, tri):
    def sign(p1, p2, p3):
        return (p1[0] - p3[0]) * (p2[1] - p3[1]) - (p2[0] - p3[0]) * (p1[1] - p3[1])

    b1 = sign(pt, tri[0], tri[1]) < 0.0
    b2 = sign(pt, tri[1], tri[2]) < 0.0
    b3 = sign(pt, tri[2], tri[0]) < 0.0

    return ((b1 == b2) and (b2 == b3))

# Callback del mouse para seleccionar puntos
def select_points(event, x, y, flags, param):
    global points, mesh_triangles
    if event == cv2.EVENT_LBUTTONDOWN and len(points) < 3:
        points.append([x, y])
        print(f"Punto {len(points)}: ({x}, {y})")
        if len(points) == 3:
            p1, p2, p3 = np.array(points[0]), np.array(points[1]), np.array(points[2])
            mesh_triangles = generate_equilateral_mesh(p1, p2, p3, rows=4, cols=6)
            print("Malla generada con", len(mesh_triangles), "triángulos")

# Inicializar cámara
cap = cv2.VideoCapture(1)
cv2.namedWindow("Triangular Workspace")
cv2.setMouseCallback("Triangular Workspace", select_points)

while True:
    ret, frame = cap.read()
    if not ret:
        break

    # Dibujar puntos seleccionados
    for i, pt in enumerate(points):
        cv2.circle(frame, tuple(pt), 5, (0, 0, 255), -1)
        cv2.putText(frame, f"P{i+1}", tuple(pt), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 255), 2)

    # Dibujar malla de triángulos
    if len(mesh_triangles) > 0:
        for tri in mesh_triangles:
            pts = np.array(tri, np.int32)
            pts = pts.reshape((-1, 1, 2))
            cv2.polylines(frame, [pts], isClosed=True, color=(255, 255, 0), thickness=1)

    # --- Detección de manzana AMARILLA ---
    hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)

    # Rango para AMARILLO tipo manzana golden
    lower_yellow = np.array([20, 100, 100])   # H, S, V mínimos
    upper_yellow = np.array([35, 255, 255])   # H, S, V máximos

    mask = cv2.inRange(hsv, lower_yellow, upper_yellow)

    # Operaciones morfológicas para limpiar ruido
    mask = cv2.erode(mask, None, iterations=2)
    mask = cv2.dilate(mask, None, iterations=2)

    # Encontrar contornos
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    obj_center = None
    if contours:
        largest_contour = max(contours, key=cv2.contourArea)
        area = cv2.contourArea(largest_contour)

        # Filtrar por área mínima (ajusta según el tamaño real en cámara)
        if area > 800:
            M = cv2.moments(largest_contour)
            if M["m00"] != 0:
                cx = int(M["m10"] / M["m00"])
                cy = int(M["m01"] / M["m00"])
                obj_center = (cx, cy)

                # Dibujar contorno y centro
                cv2.drawContours(frame, [largest_contour], -1, (0, 255, 255), 2)  # Amarillo en BGR
                cv2.circle(frame, (cx, cy), 10, (0, 255, 255), -1)
                cv2.putText(frame, "Manzana Amarilla", (cx - 70, cy - 20), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 255), 2)

    # Encontrar en qué triángulo está la manzana
    if obj_center and len(mesh_triangles) > 0:
        for i, tri in enumerate(mesh_triangles):
            if point_in_triangle(obj_center, tri):
                pts = np.array(tri, np.int32)
                pts = pts.reshape((-1, 1, 2))
                cv2.polylines(frame, [pts], isClosed=True, color=(0, 255, 255), thickness=3)
                cv2.putText(frame, f"Objetivo: Tri {i}", (10, 60), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 255), 2)
                break

    # Mostrar instrucciones
    if len(points) < 3:
        cv2.putText(frame, "Haz clic en 3 puntos para definir el primer triangulo", (10, 30),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)
    cv2.putText(frame, "Presiona 'r' para reiniciar | 'q' para salir", (10, frame.shape[0] - 20),
                cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)

    # Mostrar máscara (útil para depurar)
    cv2.imshow("Mascara Amarilla", mask)
    cv2.imshow("Triangular Workspace", frame)

    key = cv2.waitKey(1) & 0xFF
    if key == ord('q'):
        break
    elif key == ord('r'):
        points = []
        mesh_triangles = []
        print("Reiniciando selección...")

cap.release()
cv2.destroyAllWindows()

Punto 1: (182, 297)
Punto 2: (230, 299)
Punto 3: (196, 326)
Malla generada con 118 triángulos
Reiniciando selección...
Punto 1: (310, 304)
Punto 2: (336, 303)
Punto 3: (320, 334)
Malla generada con 118 triángulos
Reiniciando selección...
Punto 1: (312, 305)
Punto 2: (388, 302)
Punto 3: (347, 341)
Malla generada con 118 triángulos
Reiniciando selección...
Punto 1: (224, 250)
Punto 2: (311, 254)
Punto 3: (248, 333)
Malla generada con 118 triángulos


# PROTOTIPO 2


In [27]:
import cv2
import numpy as np
import math

# Variables globales
mesh_triangles = []    # Lista de triángulos generados
obj_center = None      # Centro del objeto detectado

# Función para generar una malla de triángulos equiláteros que cubre toda la imagen
def generate_full_mesh(frame_width, frame_height, triangle_size=60):
    """
    Genera una malla de triángulos equiláteros que cubre toda la imagen.
    triangle_size: tamaño del lado del triángulo (en píxeles)
    """
    triangles = []
    h = (triangle_size * math.sqrt(3)) / 2  # Altura del triángulo

    # Recorremos las filas
    for i in range(0, frame_height, int(h)):
        # Alternar desplazamiento horizontal entre filas
        offset = 0 if (i // int(h)) % 2 == 0 else triangle_size // 2
        
        for j in range(offset, frame_width, triangle_size):
            # Triángulo apuntando hacia arriba
            p1 = (j, i)
            p2 = (j + triangle_size, i)
            p3 = (j + triangle_size // 2, i + h)
            triangles.append([p1, p2, p3])

            # Triángulo apuntando hacia abajo (solo si hay espacio)
            if i + h < frame_height:
                p1 = (j, i + h)
                p2 = (j + triangle_size, i + h)
                p3 = (j + triangle_size // 2, i + 2*h)
                triangles.append([p1, p2, p3])

    return triangles

# Función para saber si un punto está dentro de un triángulo
def point_in_triangle(pt, tri):
    def sign(p1, p2, p3):
        return (p1[0] - p3[0]) * (p2[1] - p3[1]) - (p2[0] - p3[0]) * (p1[1] - p3[1])

    b1 = sign(pt, tri[0], tri[1]) < 0.0
    b2 = sign(pt, tri[1], tri[2]) < 0.0
    b3 = sign(pt, tri[2], tri[0]) < 0.0

    return ((b1 == b2) and (b2 == b3))

# Inicializar cámara
cap = cv2.VideoCapture(1)
frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

# Generar malla automáticamente al inicio
mesh_triangles = generate_full_mesh(frame_width, frame_height, triangle_size=60)

print(f"Malla generada con {len(mesh_triangles)} triángulos")

while True:
    ret, frame = cap.read()
    if not ret:
        break

    # Dibujar malla de triángulos
    for tri in mesh_triangles:
        pts = np.array(tri, np.int32)
        pts = pts.reshape((-1, 1, 2))
        cv2.polylines(frame, [pts], isClosed=True, color=(255, 255, 0), thickness=1)

    # --- Detección de manzana AMARILLA ---
    hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)

    # Rango HSV para amarillo (ajustado para manzana)
    lower_yellow = np.array([20, 80, 100])
    upper_yellow = np.array([35, 255, 255])

    mask = cv2.inRange(hsv, lower_yellow, upper_yellow)

    # Operaciones morfológicas
    kernel = np.ones((5, 5), np.uint8)
    mask = cv2.erode(mask, kernel, iterations=1)
    mask = cv2.dilate(mask, kernel, iterations=2)

    # Mostrar máscara
    cv2.imshow("Mascara Amarilla", mask)

    # Encontrar contornos
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    obj_center = None
    if contours:
        largest_contour = max(contours, key=cv2.contourArea)
        area = cv2.contourArea(largest_contour)

        if area > 500:  # Filtrar ruido
            M = cv2.moments(largest_contour)
            if M["m00"] != 0:
                cx = int(M["m10"] / M["m00"])
                cy = int(M["m01"] / M["m00"])
                obj_center = (cx, cy)

                # Dibujar centro y contorno
                cv2.drawContours(frame, [largest_contour], -1, (0, 255, 255), 2)
                cv2.circle(frame, (cx, cy), 10, (0, 255, 255), -1)
                cv2.putText(frame, "Manzana", (cx - 30, cy - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 255), 2)

    # Resaltar triángulo donde está el objeto
    if obj_center and len(mesh_triangles) > 0:
        for i, tri in enumerate(mesh_triangles):
            if point_in_triangle(obj_center, tri):
                pts = np.array(tri, np.int32)
                pts = pts.reshape((-1, 1, 2))
                cv2.polylines(frame, [pts], isClosed=True, color=(0, 255, 255), thickness=3)
                cv2.putText(frame, f"Objetivo: Tri {i}", (10, 60), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 255), 2)
                break

    # Mostrar instrucciones
    cv2.putText(frame, "Presiona 'q' para salir", (10, frame.shape[0] - 20),
                cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)

    cv2.imshow("Triangular Workspace", frame)

    key = cv2.waitKey(1) & 0xFF
    if key == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

Malla generada con 209 triángulos


# PROTOTIPO 3 (el mejor)

In [5]:
import cv2
import numpy as np
import math

# variables globales
points = []          
mesh_triangles = []  
obj_center = None    

# generar malla de triangulos equilateros dentro del rectangulo de 4 puntos
def generate_mesh_in_rect(rect, triangle_size=60):
    triangles = []
    h = (triangle_size * math.sqrt(3)) / 2  # altura del triángulo equilátero

    # Ordenar los puntos para formar un rectángulo
    sorted_points = sorted(rect, key=lambda p: (p[0], p[1]))
    x_min = min(p[0] for p in sorted_points)
    x_max = max(p[0] for p in sorted_points)
    y_min = min(p[1] for p in sorted_points)
    y_max = max(p[1] for p in sorted_points)

    step_y = int(h)  # espacio vertical entre filas
    step_x = triangle_size // 2  

    # Recorremos las filas verticalmente
    for i in range(y_min, y_max - int(h) + 1, step_y):
        # Recorremos horizontalmente con paso de triangle_size//2
        for j in range(x_min, x_max - triangle_size + 1, step_x):
            # Alternar entre triángulo hacia arriba y hacia abajo
            is_up = (j // step_x) % 2 == 0

            if is_up:
                # ▲ Triángulo apuntando hacia arriba
                p1 = (j, i)
                p2 = (j + triangle_size, i)
                p3 = (j + triangle_size // 2, i + h)
            else:
                # ▼ Triángulo apuntando hacia abajo
                p1 = (j, i + h)
                p2 = (j + triangle_size, i + h)
                p3 = (j + triangle_size // 2, i)

            # Verificar que todos los vértices estén dentro del rectángulo
            if (x_min <= p1[0] <= x_max and y_min <= p1[1] <= y_max and
                x_min <= p2[0] <= x_max and y_min <= p2[1] <= y_max and
                x_min <= p3[0] <= x_max and y_min <= p3[1] <= y_max):
                triangles.append([p1, p2, p3])

    return triangles

# Función para saber si un punto está dentro de un triángulo
def point_in_triangle(pt, tri):
    def sign(p1, p2, p3):
        return (p1[0] - p3[0]) * (p2[1] - p3[1]) - (p2[0] - p3[0]) * (p1[1] - p3[1])

    b1 = sign(pt, tri[0], tri[1]) < 0.0
    b2 = sign(pt, tri[1], tri[2]) < 0.0
    b3 = sign(pt, tri[2], tri[0]) < 0.0

    return ((b1 == b2) and (b2 == b3))

# Callback del mouse para seleccionar puntos
def select_points(event, x, y, flags, param):
    global points, mesh_triangles  # ← Aquí es clave: declarar como global
    if event == cv2.EVENT_LBUTTONDOWN and len(points) < 4:
        points.append([x, y])
        print(f"Punto {len(points)}: ({x}, {y})")
        if len(points) == 4:
            # Generar malla dentro del rectángulo
            mesh_triangles = generate_mesh_in_rect(points, triangle_size=60)
            print("Malla generada con", len(mesh_triangles), "triángulos")

# Inicializar cámara
cap = cv2.VideoCapture(1)
cv2.namedWindow("Triangular Workspace")
cv2.setMouseCallback("Triangular Workspace", select_points)

while True:
    ret, frame = cap.read()
    if not ret:
        break

    # Dibujar puntos seleccionados
    for i, pt in enumerate(points):
        cv2.circle(frame, tuple(pt), 5, (0, 0, 255), -1)
        cv2.putText(frame, f"P{i+1}", tuple(pt), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 255), 2)

    # Dibujar malla de triángulos
    if len(mesh_triangles) > 0:
        for tri in mesh_triangles:
            pts = np.array(tri, np.int32)
            pts = pts.reshape((-1, 1, 2))
            cv2.polylines(frame, [pts], isClosed=True, color=(255, 255, 0), thickness=1)

    # --- Detección de manzana AMARILLA ---
    hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)

    # Rango HSV para amarillo (ajustado para manzana)
    lower_yellow = np.array([20, 80, 100])
    upper_yellow = np.array([35, 255, 255])

    mask = cv2.inRange(hsv, lower_yellow, upper_yellow)

    # Operaciones morfológicas
    kernel = np.ones((5, 5), np.uint8)
    mask = cv2.erode(mask, kernel, iterations=1)
    mask = cv2.dilate(mask, kernel, iterations=2)

    # Mostrar máscara
    cv2.imshow("Mascara Amarilla", mask)

    # Encontrar contornos
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    obj_center = None
    if contours:
        largest_contour = max(contours, key=cv2.contourArea)
        area = cv2.contourArea(largest_contour)

        if area > 500:  # Filtrar ruido
            M = cv2.moments(largest_contour)
            if M["m00"] != 0:
                cx = int(M["m10"] / M["m00"])
                cy = int(M["m01"] / M["m00"])
                obj_center = (cx, cy)

                # Dibujar centro y contorno
                cv2.drawContours(frame, [largest_contour], -1, (0, 255, 255), 2)
                cv2.circle(frame, (cx, cy), 10, (0, 255, 255), -1)
                cv2.putText(frame, "Manzana", (cx - 30, cy - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 255), 2)

    # Resaltar triángulo donde está el objeto
    if obj_center and len(mesh_triangles) > 0:
        for i, tri in enumerate(mesh_triangles):
            if point_in_triangle(obj_center, tri):
                pts = np.array(tri, np.int32)
                pts = pts.reshape((-1, 1, 2))
                cv2.polylines(frame, [pts], isClosed=True, color=(0, 255, 255), thickness=3)
                cv2.putText(frame, f"Objetivo: Tri {i}", (10, 60), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 255), 2)
                break

    # Mostrar instrucciones
    if len(points) < 4:
        cv2.putText(frame, "Haz clic en 4 puntos para definir el área de trabajo", (10, 30),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)
    cv2.putText(frame, "Presiona 'q' para salir", (10, frame.shape[0] - 20),
                cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)

    cv2.imshow("Triangular Workspace", frame)

    key = cv2.waitKey(1) & 0xFF
    if key == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

Punto 1: (136, 185)
Punto 2: (136, 337)
Punto 3: (452, 319)
Punto 4: (445, 114)
Malla generada con 36 triángulos


# prototipo 3 con homografia

In [ ]:
import cv2
import numpy as np
import math

points = []          
mesh_triangles = []  
obj_center = None    
H = None             # Homografía
H_inv = None         # Homografía inversa
rectified_width, rectified_height = 350, 200  # Tamaño del espacio rectificado (menor numero triangulos mas grandes)

def generate_mesh_in_rect(triangle_size=60):
    triangles = []
    h = (triangle_size * math.sqrt(3)) / 2  # altura del triángulo equilátero (float)

    step_y = int(h)          # paso vertical en píxeles (entero)
    step_x = triangle_size // 2  # paso horizontal (entero)

    # Asegurar que no excedemos los límites del rectángulo
    max_i = int(rectified_height - h)
    if max_i < 0:
        max_i = 0

    max_j = rectified_width - triangle_size
    if max_j < 0:
        max_j = 0

    for i in range(0, max_i + 1, step_y):
        row_index = i // step_y  # índice de fila

        for j in range(0, max_j + 1, step_x):
            col_index = j // step_x  # índice de columna

            # Alternar orientación: si (fila + columna) es par entonces triángulo hacia arriba
            is_up = ((row_index + col_index) % 2 == 0)

            if is_up:
                p1 = (j, i)
                p2 = (j + triangle_size, i)
                p3 = (j + triangle_size // 2, i + h)
            else:
                p1 = (j, i + h)
                p2 = (j + triangle_size, i + h)
                p3 = (j + triangle_size // 2, i)

            # Verificar que los puntos estén dentro del rectángulo rectificado
            if (0 <= p1[0] <= rectified_width and 0 <= p1[1] <= rectified_height and
                0 <= p2[0] <= rectified_width and 0 <= p2[1] <= rectified_height and
                0 <= p3[0] <= rectified_width and 0 <= p3[1] <= rectified_height):
                triangles.append([p1, p2, p3])

    return triangles

def point_in_triangle(pt, tri):
    #verificacion del punto respecto a cada lado del triangulo
    def sign(p1, p2, p3):
        return (p1[0] - p3[0]) * (p2[1] - p3[1]) - (p2[0] - p3[0]) * (p1[1] - p3[1])

    b1 = sign(pt, tri[0], tri[1]) < 0.0
    b2 = sign(pt, tri[1], tri[2]) < 0.0
    b3 = sign(pt, tri[2], tri[0]) < 0.0

    return ((b1 == b2) and (b2 == b3))

# Callback del mouse para seleccionar puntos
def select_points(event, x, y, flags, param):
    global points, mesh_triangles, H, H_inv
    if event == cv2.EVENT_LBUTTONDOWN and len(points) < 4:
        points.append([x, y])
        print(f"Punto {len(points)}: ({x}, {y})")
        if len(points) == 4:
            # Ordenar puntos: queremos [superior izq, superior der, inferior der, inferior izq]
            pts = np.array(points, dtype=np.float32)
            # Ordenar por: arriba izq, arriba der, abajo der, abajo izq
            # Usamos: suma x+y para esquina inferior derecha, diferencia y-x para superior derecha, etc.
            s = pts.sum(axis=1)
            diff = np.diff(pts, axis=1)
            rect = np.zeros((4, 2), dtype=np.float32)
            rect[0] = pts[np.argmin(s)]      # arriba izq
            rect[2] = pts[np.argmax(s)]      # abajo der
            rect[1] = pts[np.argmin(diff)]   # arriba der
            rect[3] = pts[np.argmax(diff)]   # abajo izq

            # Definir puntos destino (rectángulo frontal)
            dst = np.array([
                [0, 0],
                [rectified_width, 0],
                [rectified_width, rectified_height],
                [0, rectified_height]
            ], dtype=np.float32)

            # Calcular homografía: de rectificado -> imagen
            H = cv2.getPerspectiveTransform(dst, rect)
            H_inv = cv2.getPerspectiveTransform(rect, dst)  # para mapear puntos de imagen a rectificado

            # Generar malla en espacio rectificado
            mesh_rect = generate_mesh_in_rect(triangle_size=60)

            # Proyectar cada triángulo al espacio de la imagen usando H
            mesh_triangles = []
            for tri in mesh_rect:
                tri_np = np.array(tri, dtype=np.float32).reshape(-1, 1, 2)
                tri_warped = cv2.perspectiveTransform(tri_np, H)
                tri_warped = [tuple(pt[0]) for pt in tri_warped]  # Convertir a lista de tuplas
                mesh_triangles.append(tri_warped)

            print("Malla generada con", len(mesh_triangles), "triángulos (aplicada homografía)")

cap = cv2.VideoCapture(1) # camara usb
cv2.namedWindow("Triangular Workspace")
cv2.setMouseCallback("Triangular Workspace", select_points)

while True:
    ret, frame = cap.read()
    if not ret:
        break

    # dibujado puntos seleccionados
    for i, pt in enumerate(points):
        cv2.circle(frame, tuple(pt), 5, (0, 0, 255), -1)
        cv2.putText(frame, f"P{i+1}", tuple(pt), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 255), 2)

    # dibujado malla de triángulos (ya deformados por homografía)
    if len(mesh_triangles) > 0:
        for tri in mesh_triangles:
            pts = np.array(tri, np.int32)
            pts = pts.reshape((-1, 1, 2))
            cv2.polylines(frame, [pts], isClosed=True, color=(255, 255, 0), thickness=1)

    # --- Detección de manzana AMARILLA ---
    hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)
    lower_yellow = np.array([20, 80, 100])
    upper_yellow = np.array([35, 255, 255])
    mask = cv2.inRange(hsv, lower_yellow, upper_yellow)

    # Operaciones morfológicas
    kernel = np.ones((5, 5), np.uint8)
    mask = cv2.erode(mask, kernel, iterations=1)
    mask = cv2.dilate(mask, kernel, iterations=2)
    cv2.imshow("Mascara Amarilla", mask)
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    obj_center = None
    if contours:
        largest_contour = max(contours, key=cv2.contourArea)
        area = cv2.contourArea(largest_contour)

        if area > 500:  # Filtrar ruido
            M = cv2.moments(largest_contour)
            if M["m00"] != 0:
                cx = int(M["m10"] / M["m00"])
                cy = int(M["m01"] / M["m00"])
                obj_center = (cx, cy)

                # dibujado centro y contorno
                cv2.drawContours(frame, [largest_contour], -1, (0, 255, 255), 2)
                cv2.circle(frame, (cx, cy), 10, (0, 255, 255), -1)
                cv2.putText(frame, "Manzana", (cx - 30, cy - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 255), 2)

    # resaltar triángulo donde está el objeto (en espacio rectificado)
    if obj_center and H_inv is not None and len(mesh_triangles) > 0:
        # Mapear punto de imagen al espacio rectificado
        pt_np = np.array([[obj_center]], dtype=np.float32)
        pt_rect = cv2.perspectiveTransform(pt_np, H_inv)
        cx_rect, cy_rect = pt_rect[0][0]

        # Buscar en qué triángulo del espacio rectificado cae
        mesh_rect = generate_mesh_in_rect(triangle_size=60)  # Generar malla original (rectificada)
        for i, tri in enumerate(mesh_rect):
            if point_in_triangle((cx_rect, cy_rect), tri):
                # Dibujar el triángulo correspondiente en la imagen (usando mesh_triangles[i])
                tri_img = mesh_triangles[i]
                pts = np.array(tri_img, np.int32).reshape((-1, 1, 2))
                cv2.polylines(frame, [pts], isClosed=True, color=(0, 255, 255), thickness=3)
                cv2.putText(frame, f"Objetivo: Tri {i}", (10, 60), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 255), 2)
                break

    # mostrar textos
    if len(points) < 4:
        cv2.putText(frame, "Haz clic en 4 puntos para definir el área de trabajo", (10, 30),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)
        
    cv2.putText(frame, "Presiona 'q' para salir", (10, frame.shape[0] - 20),
                cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)
    cv2.putText(frame, "Presiona 'r' para reiniciar", (10, frame.shape[0] - 40),
                cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)

    cv2.imshow("Triangular Workspace", frame)

    # q quitar r reiniciar
    key = cv2.waitKey(1) & 0xFF
    if key == ord('q'):
        break
    elif key == ord('r'): 
        points = []
        mesh_triangles = []
        H = None
        H_inv = None
        print("Selección reiniciada")

cap.release()
cv2.destroyAllWindows()

Punto 1: (157, 105)
Punto 2: (168, 376)
Punto 3: (492, 372)
Punto 4: (442, 85)
Malla generada con 30 triángulos (aplicada homografía)
Selección reiniciada
Punto 1: (136, 228)
Punto 2: (17, 298)
Punto 3: (526, 314)
Punto 4: (481, 218)
Malla generada con 30 triángulos (aplicada homografía)
Selección reiniciada
Punto 1: (134, 213)
Punto 2: (12, 298)
Punto 3: (510, 319)
Punto 4: (472, 219)
Malla generada con 30 triángulos (aplicada homografía)
Selección reiniciada
Punto 1: (132, 231)
Punto 2: (9, 364)
Punto 3: (606, 366)
Punto 4: (556, 250)
Malla generada con 30 triángulos (aplicada homografía)
Selección reiniciada
Punto 1: (124, 178)
Punto 2: (3, 390)
Punto 3: (632, 375)
Punto 4: (554, 179)
Malla generada con 30 triángulos (aplicada homografía)
